# 04 — Szövegelemzés és klasszikus modellek a BERT-címkéken / Text analysis and classical models on the BERT labels

HU: A konzulens által jónak ítélt TF-IDF-alapú szóelemzés most a **szövegből származó** címkéken fut (`label_final`),
nem a pontszám-vágáson. Utána a klasszikus modellek (NB, LogReg, SVM) megtanulják a BERT-címkéket: gyors, értelmezhető
"tanítvány" modellek, amelyek CPU-n is címkéznek.

EN: The TF-IDF word analysis (approved by the supervisor) now runs on the **text-derived** labels (`label_final`),
not on the score cut. Then the classical models (NB, LogReg, SVM) learn the BERT labels: fast, interpretable
"student" models that label on a CPU too.

In [ ]:
# HU: A notebooks/ mappából futunk, ezért a szülőmappát a path-hoz adjuk (src csomag).
# EN: We run from the notebooks/ folder, so the parent folder is added to the path (src package).
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import set_seed
from src.config import CFG, PATHS, SENTIMENT_COLORS, INK, APPROACHES, apply_plot_style
from src import data_loading as dl
from src import preprocessing as pp
from src import evaluation as ev
from src import descriptives as ds
from src import text_analysis as ta
from src import aspects as asp
from src import models_classical as mc
from src import label_validation as lv
from src import pipeline
import time

# HU: Reprodukálhatóság + egységes ábrastílus. / EN: Reproducibility + uniform figure style.
set_seed()
PATHS.ensure()
apply_plot_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

LABEL_COL = "label_final"
data = ds.prepare_star_rating(dl.read_frame(pipeline.LABELLED))
print(f"{len(data):,} reviews | label: {LABEL_COL}")
display(data[LABEL_COL].value_counts(normalize=True).round(3).to_frame("share"))

## 1. Megkülönböztető kifejezések / Distinctive terms

In [ ]:
dist_terms = ta.distinctive_terms(data["text_clean"], data[LABEL_COL], ngram_range=(1, 2), top_n=15, min_count=30)
ev.save_table(dist_terms, "04_distinctive_terms")
ta.plot_distinctive_terms(dist_terms, save_as="04_distinctive_terms"); plt.show()

multi = ta.distinctive_terms(data["text_clean"], data[LABEL_COL], ngram_range=(2, 3), top_n=15, min_count=30)
ev.save_table(multi, "04_distinctive_multiword")
ta.plot_distinctive_terms(multi, save_as="04_distinctive_multiword"); plt.show()

In [ ]:
neg_bi = ta.negation_bigrams(data["text_clean"], data[LABEL_COL], min_count=20)
display(neg_bi); ev.save_table(neg_bi, "04_negation_bigrams")

neg_effect = ta.compare_with_without_negation(data["text_clean"], data[LABEL_COL])
display(neg_effect); ev.save_table(neg_effect, "04_negation_effect")
ta.plot_negation_effect(neg_effect, save_as="04_negation_effect"); plt.show()

ngram_cmp = ta.ngram_level_comparison(data["text_clean"], data[LABEL_COL], min_count=30)
display(ngram_cmp); ev.save_table(ngram_cmp, "04_ngram_levels")

## 2. Leíró statisztika és kereszttáblák / Descriptive statistics and crosstabs

In [ ]:
overview = ds.dataset_overview(data, label_col=LABEL_COL)
display(overview); ev.save_table(overview, "04_dataset_overview_labelled")

crosstabs = ds.run_all_crosstabs(data, min_count=100, label_col=LABEL_COL)
for col, tab in crosstabs.items():
    print(f"=== {col} ==="); display(tab)

In [ ]:
def heatmap(rows, cols, xlabel, ylabel, fname, min_count=50):
    """HU: Hőtérkép hibatűréssel. / EN: Heatmap with fault tolerance."""
    if not {rows, cols} <= set(data.columns):
        return print(f"Skipped — missing column: {rows}, {cols}")
    try:
        lift, counts, base = ds.negativity_heatmap(data, rows, cols, min_count=min_count, label_col=LABEL_COL)
    except ValueError as e:
        return print(f"Skipped: {e}")
    ev.save_table(lift, fname)
    ds.plot_negativity_heatmap(lift, counts, base, "Share of negative reviews relative to the average",
                               xlabel=xlabel, ylabel=ylabel, save_as=fname)
    plt.show()

heatmap("accommodation_type", "accommodation_star_rating", "Star rating", "Accommodation type", "04_heatmap_type_x_stars")
heatmap("accommodation_country", "accommodation_type", "Accommodation type", "Country", "04_heatmap_country_x_type")

## 3. Aspektusok / Aspects

HU: Két mondatszintű forrás: (1) a Booking-mező polaritása (a vendég döntése), (2) ha elérhető, a (B) modell mondatszintű
előrejelzése. A kettő összevetése maga is ellenőrzés.
EN: Two sentence-level sources: (1) the Booking field polarity (the guest's choice), (2) if available, the sentence-level
predictions of model (B). Comparing the two is a check in itself.

In [ ]:
field_sent = asp.add_aspect_columns(asp.explode_by_polarity(data))
aspect_field = asp.aspect_sentiment_table(field_sent, label_col="field_polarity", min_count=50)
display(aspect_field); ev.save_table(aspect_field, "04_aspects_by_field_polarity")
asp.plot_aspect_sentiment(aspect_field, "Aspects — Booking field polarity", save_as="04_aspects_by_field_polarity")
plt.show()

if dl.frame_exists(PATHS.labels / "sentences_B"):
    sent_b = asp.add_aspect_columns(dl.read_frame(PATHS.labels / "sentences_B"))
    aspect_b = asp.aspect_sentiment_table(sent_b, label_col="B_label", min_count=50)
    display(aspect_b); ev.save_table(aspect_b, "04_aspects_by_bert_sentence")
    asp.plot_aspect_sentiment(aspect_b, "Aspects — BERT (B) sentence labels", save_as="04_aspects_by_bert_sentence")
    plt.show()

## 4. Klasszikus "tanítvány" modellek / Classical "student" models

HU: A mért pontosság itt a **BERT-címkékkel való egyezés** (tudásdesztilláció). A valódi pontosságot a gold mintán
mérjük (5. pont). `TRAIN_ON_UNANIMOUS=True`: csak az egybehangzó (legbiztosabb) címkéken tanít.
EN: The measured accuracy here is **agreement with the BERT labels** (knowledge distillation). True accuracy is measured
on the gold sample (section 5). `TRAIN_ON_UNANIMOUS=True`: trains only on the unanimous (most reliable) labels.

In [ ]:
TRAIN_ON_UNANIMOUS = False
train_df, val_df, test_df = pp.train_val_test_split(data, label_col=LABEL_COL)
if TRAIN_ON_UNANIMOUS and "unanimous" in train_df:
    train_df = train_df[train_df["unanimous"]]

display(ev.baseline_metrics(test_df[LABEL_COL]))

t0 = time.time()
summary, fitted = mc.train_and_evaluate_all(train_df, val_df, label_col=LABEL_COL)
print(f"Training time: {time.time() - t0:.1f} s")
display(summary[["accuracy", "f1_macro", "f1_weighted", "n_features"]].round(4))
ev.save_table(summary, "04_classical_validation")

In [ ]:
best_name = summary.index[0]
y_true = test_df[LABEL_COL].astype(str)
y_pred = fitted[best_name].predict(test_df["text_clean"])
print(f"Best on validation: {best_name}")
display(ev.per_class_report(y_true, y_pred).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, name in zip(axes, mc.MODEL_NAMES):
    ev.plot_confusion_matrix(y_true, fitted[name].predict(test_df["text_clean"]), title=name, ax=ax)
plt.tight_layout(); ev.save_figure("04_confusion_matrices_classical"); plt.show()

In [ ]:
# HU: Ablációs vizsgálat mintán (8 változat). / EN: Ablation study on a sample (8 variants).
abl_train = train_df.sample(min(20_000, len(train_df)), random_state=CFG.random_seed)
ablation = mc.preprocessing_ablation(abl_train, val_df, label_col=LABEL_COL)
display(ablation[["accuracy", "f1_macro", "n_features"]].round(4))
ev.save_table(ablation, "04_preprocessing_ablation")

In [ ]:
top = mc.top_features_per_class(fitted["logreg"], top_n=15)
ev.save_table(top, "04_top_features_logreg")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, label in zip(axes, CFG.labels):
    sub = top[top["sentiment"] == label].sort_values("weight")
    ax.barh(sub["feature"], sub["weight"], color=SENTIMENT_COLORS[label])
    ax.set_title(label, loc="left"); ax.set_xlabel("Logistic regression weight")
    ax.grid(axis="y", visible=False)
plt.tight_layout(); ev.save_figure("04_top_features_logreg"); plt.show()

errors = ev.error_analysis(test_df, y_pred, label_col=LABEL_COL, n_examples=5)
ev.save_table(errors, "04_error_examples")
errors.head(10)

## 5. Összevetés a gold mintán / Comparison on the gold sample

In [ ]:
gold_file = PATHS.annotation / "gold_sample_to_annotate.csv"
if gold_file.exists() and len(lv.load_gold()):
    gold = lv.load_gold()
    gold_data = data.set_index("review_id").loc[lambda d: d.index.isin(gold.index)].copy()
    gold_data[f"tfidf_{best_name}"] = fitted[best_name].predict(gold_data["text_clean"])
    sources = {"BERT final": "label_final", f"TF-IDF {best_name}": f"tfidf_{best_name}"}
    comparison = lv.evaluate_on_gold(gold, gold_data, sources)
    display(comparison); ev.save_table(comparison, "04_gold_bert_vs_tfidf")
else:
    print("No annotated gold sample yet (see notebook 03, section 5).")